<img src="https://drive.google.com/uc?export=view&id=1Q6vQcIWFPY27isBepABpJ7nroUNKox_Z" width="100%">


In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch

In [2]:
def get_oxford_pet_loaders(data_dir="./data", batch_size=32, 
                           val_split=0.2, num_workers=1):
    """
    Crea DataLoaders de entrenamiento y validación para Oxford-IIIT Pet (cats vs dogs).

    Args:
        data_dir (str): Carpeta donde se descargará/guardará el dataset.
        batch_size (int): Tamaño de batch para los loaders.
        val_split (float): Proporción del set de entrenamiento que se usará para validación.
        num_workers (int): Número de procesos de carga de datos (ajusta según CPU).
        seed (int): Semilla para reproducibilidad.

    Returns:
        train_loader, val_loader, class_names
    """

    # Transformaciones para VGG
    transform = transforms.Compose([
        transforms.Resize((224, 224)),        
        transforms.RandomHorizontalFlip(p=0.5),  
        transforms.ToTensor(),                 
        transforms.Normalize(           
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225])])

    dataset = datasets.OxfordIIITPet(root=data_dir,
                                     split="trainval",
                                     target_types="category",  # 0=cat, 1=dog
                                     download=True,transform=transform)

    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size

    torch_generator = torch.Generator()
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size],generator=torch_generator)

    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=num_workers)

    class_names = ["cat", "dog"]

    return train_loader, val_loader, class_names

In [3]:
train_loader, val_loader, class_names = get_oxford_pet_loaders()

URLError: <urlopen error [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión>

In [ ]:
!pip install unidecode

In [ ]:
!python -m spacy download es_core_news_sm

In [29]:
import pandas as pd
import re
import unicodedata
from unidecode import unidecode
import pandas as pd
import spacy

In [61]:
corpus = pd.read_csv('/content/corpus_completo.csv')
corpus

,fecha,titulo,cuerpo,url,categoria,fuente,categoria_agrupada
0,2025-08-26,YouTube hizo inesperado movimiento con intelig...,La plataforma de entretenimiento YouTube cuent...,https://www.semana.com//tecnologia/articulo/yo...,tecnologia,Semana,tecnologia_ciencia
1,2025-08-26,Apple reveló la fecha de lanzamiento del iPhon...,Apple celebrará un nuevo evento de presentació...,https://www.semana.com//tecnologia/articulo/ap...,tecnologia,Semana,tecnologia_ciencia
2,2025-08-21,La estrella más lejana conocida puede no serlo...,"Eärendel, un objeto descubierto en 2022 con el...",https://www.semana.com//tecnologia/articulo/la...,tecnologia,Semana,tecnologia_ciencia
3,2025-08-08,Por primera vez en la historia: investigadores...,Un equipo de especialistas en ciberseguridad r...,https://www.semana.com//tecnologia/articulo/po...,tecnologia,Semana,tecnologia_ciencia
4,2025-08-28,No solo es la inteligencia artificial: esta es...,"Para muchos ha resultado sorprendente cómo, en...",https://www.semana.com//tecnologia/articulo/no...,tecnologia,Semana,tecnologia_ciencia
...,...,...,...,...,...,...,...
1816,2025-05-02T23:39:43.817Z,Festival Jazz Libre 2025: una celebración de r...,NaN,https://www.elespectador.com/servicios/festiva...,Actualidad,El Espectador,entretenimiento
1817,2025-05-02T15:42:30.078Z,"media maratón de Bogotá 2025: inscripciones, r...",La media maratón de Bogotá (mmB) regresa en 20...,https://www.elespectador.com/servicios/vive-la...,Actualidad,El Espectador,entretenimiento
1818,2025-04-29T17:49:03.441Z,Astronauta de la NASA y Rigoberto Urán compart...,NaN,https://www.elespectador.com/servicios/astrona...,Actualidad,El Espectador,entretenimiento
1819,2025-04-27T13:00:00Z,“La emergencia sanitaria es una medida adecuad...,"Entrevista a Zulma Cucunubá, epidemióloga doct...",https://www.elespectador.com/salud/la-emergenc...,Actualidad,El Espectador,entretenimiento


In [49]:

colombiano = pd.read_csv('/content/ElColombiano_Estatico.csv')
semana = pd.read_parquet('/content/semana_news_202508.parquet')
espectador = pd.read_excel('/content/noticias_elespectador (1).xlsx')

colombiano = colombiano.rename(columns={
    "hora": "fecha",
    "titulo": "titulo",
    "cuerpo": "cuerpo",
    "url": "url",
    "categoria": "categoria"})

semana = semana.rename(columns={
    "datePublished": "fecha",
    "headline": "titulo",
    "articleBody": "cuerpo",
    "sourceUrl": "url",
    "scrappedCategory": "categoria"})

espectador = espectador.rename(columns={
    "Fecha": "fecha",
    "Titulo": "titulo",
    "Cuerpo": "cuerpo",
    "Link": "url",
    "Categoria": "categoria"})

colombiano = colombiano[["fecha", "titulo", "cuerpo", "url", "categoria"]]
semana = semana[["fecha", "titulo", "cuerpo", "url", "categoria"]]
espectador = espectador[["fecha", "titulo", "cuerpo", "url", "categoria"]]
noticias = pd.concat([colombiano, semana, espectador], ignore_index=True)
noticias["fecha"] = pd.to_datetime(noticias["fecha"], errors="coerce")

mapping = {
    "deportes": "deportes",
    "Deportes": "deportes",
    "futbol": "deportes",
    "formula-1": "deportes",
    "atletico-nacional": "deportes",
    "independiente-medellin": "deportes",


    "politica": "politica_gobierno",
    "Política": "politica_gobierno",
    "nacion": "politica_gobierno",
    "seguridad": "politica_gobierno",
    "paz-y-derechos-humanos": "politica_gobierno",
    "judicial": "politica_gobierno",
    "Judicial": "politica_gobierno",

    "gente": "sociedad_cultura",
    "cultura": "sociedad_cultura",
    "opinion": "sociedad_cultura",
    "educacion": "sociedad_cultura",
    "Educación": "sociedad_cultura",
    "Educacion": "sociedad_cultura",
    "literatura": "sociedad_cultura",
    "musica": "sociedad_cultura",
    "mascotas": "sociedad_cultura",
    "El Magazín Cultural": "sociedad_cultura",
    "Género y Diversidad": "sociedad_cultura",

    "mundo": "internacional",
    "Mundo": "internacional",
    "como": "internacional",

    "tecnologia": "tecnologia_ciencia",
    "Tecnología": "tecnologia_ciencia",
    "Tecnologia": "tecnologia_ciencia",
    "gadgets": "tecnologia_ciencia",
    "ciencia": "tecnologia_ciencia",
    "Ciencia": "tecnologia_ciencia",
    "videojuegos": "tecnologia_ciencia",
    "aplicaciones": "tecnologia_ciencia",
    "Investigación": "tecnologia_ciencia",

    "finanzas": "economia_negocios",
    "economia": "economia_negocios",
    "Economía": "economia_negocios",
    "negocios": "economia_negocios",
    "empresas": "economia_negocios",
    "agro": "economia_negocios",

    "semana-tv": "entretenimiento",
    "tv": "entretenimiento",
    "television": "entretenimiento",
    "cine": "entretenimiento",
    "moda": "entretenimiento",
    "tendencias": "entretenimiento",
    "entretenimiento": "entretenimiento",
    "Entretenimiento": "entretenimiento",
    "farandula": "entretenimiento",
    "loterias": "entretenimiento",
    "trending-topic": "entretenimiento",
    "actualidad": "entretenimiento",
    "Actualidad": "entretenimiento",
    "Reportajes": "entretenimiento",

    "turismo": "estilo_vida",
    "hablan-las-marcas": "estilo_vida",
    "sostenible": "estilo_vida",
    "medio-ambiente": "estilo_vida",
    "Ambiente": "estilo_vida",
    "movilidad": "estilo_vida",
    "obras": "estilo_vida",
    "salud": "estilo_vida",
    "Salud": "estilo_vida",
    "motores": "estilo_vida",
    "vehiculos": "estilo_vida",

    "confidenciales": "confidenciales_especiales",
    "especiales": "confidenciales_especiales",
    "foros-semana/foros-anteriores": "confidenciales_especiales",
    "mejor-colombia": "confidenciales_especiales",

    "antioquia": "regiones",
    "medellin": "regiones",
    "colombia": "regiones",
    "Colombia": "regiones",
    "Bogotá": "regiones"}

noticias["categoria_grupo"] = noticias["categoria"].map(mapping)
faltantes = sorted(set(noticias["categoria"]) - set(mapping.keys()))
print("Categorias sin mapping:", faltantes)
noticias = noticias.dropna()
noticias.shape

Categorias sin mapping: []


/tmp/ipython-input-3638805911.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  noticias["fecha"] = pd.to_datetime(noticias["fecha"], errors="coerce")
/tmp/ipython-input-3638805911.py:30: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  noticias["fecha"] = pd.to_datetime(noticias["fecha"], errors="coerce")


(6511, 6)

# Extracción de Características

---

Este notebook es una plantilla que le puede servir como guía para el tercer entregable del proyecto aplicado.


## **1. Selección del Embedding**

---

Seleccione el tipo de **embedding** que considere más apropiado para su problema. Recuerde las siguientes consideraciones:

- **Bolsas de palabras**: son útiles para representar documentos completos. Funcionan bien con corpus pequeños (<10000 documentos).
- **TF-IDF**: permite obtener una representación ponderada de documentos completos; es una alternativa a las bolsas de palabras.
- **Bolsas de N-grams**: a nivel de carácter son útiles para representar palabras o segmentos cortos de texto, funcionan bien con corpus pequeños (<10000 documentos).
- **Word2Vec**: permite representar palabras de forma semántica, requiere corpus grandes para un buen resultado (>10000 documentos).
- **FastText**: permite representar palabras o segmentos cortos de texto, funcionan bien con corpus grandes (>10000 documentos). Es una alternativa a word2vec que puede representar palabras fuera del vocabulario.
- **Doc2Vec**: permite representar documentos completos de forma semántica. Funciona bien con corpus grandes (>10000 documentos).

Justifique la selección del embedding:


El corpus que armamos contiene **6.511 documentos de noticias** de diferentes periódicos.  
Teniendo este corpus en mente, consideramos que la aproximación más robusta es **TF-IDF con n-grams** (unigramas y bigramas de palabras + n-grams de caracteres), en lugar de una BoW simple.  
Más allá del tamaño del corpus, esta elección se justifica por:

- **Señal vs. ruido:** TF-IDF controla términos poco informativos y realza indicios discriminativos.
- **Captura de contexto corto:** los n-grams recogen colocalizaciones periodísticas clave (“sube tasa”, “alerta sanitaria”, “récord histórico”) que BoW pierde.
- **Robustez léxica:** los n-grams de caracteres mitigan OOV, tildes y variantes (“inflación/Inflación”, “petróleo/petroleo”).
- **Explicabilidad y control de sobreajuste:** con clasificadores lineales (SVM/LogReg) obtenemos pesos interpretables y buena generalización en dominios específicos.
- **Costo/beneficio:** frente a embeddings preentrenados (p. ej., fastText o BERT), TF-IDF+n-grams evita desajustes de dominio y demanda computacional alta, sirviendo como un _baseline_ fuerte y auditable.

Por otro lado, el **análisis de sentimiento** que queremos realizar sobre el corpus puede generar dudas, debido a que se necesitan _embeddings_ para entrenar cualquier modelo "estable" de NLP.  
Es por esto que emplearemos un **modelo preentrenado de análisis de sentimiento** que ya incorpora _embeddings contextuales_ (**BETO** en español).  
Esto nos permite aprovechar representaciones semánticas de alta calidad aprendidas sobre corpus masivos, garantizando mejor cobertura léxica.  
Así, el uso de _embeddings_ queda implícito en la arquitectura del modelo y nos concentramos únicamente en la **etapa de inferencia con BETO**, complementando con la implementación de **TF-IDF con n-grams** para nuestros propios modelos.


## Preprocecamiento corpus


In [50]:
def preprocess_text_column(df,input_col,output_col = "texto_tfidf",
    spacy_model = "es_core_news_sm",
    remove_accents = True,remove_numbers = True,
    remove_punct = True,remove_stopwords = True,
    min_token_len = 2,batch_size = 256,n_process = 1, return_tokens = False):
    """
    Preprocesa texto en español:
    1) Limpieza inicial con regex (URLs, @menciones, #hashtags, emails, caracteres raros).
    2) Normalización de tildes/acentos (opcional).
    3) Lematización con spaCy y filtrado (stopwords, signos, números, longitud mínima).
    4) Devuelve una nueva columna con tokens listos para TF-IDF.

    Parámetros clave:
    - input_col: columna de entrada con texto crudo.
    - output_col: nueva columna con el texto procesado.
    - return_tokens: si True, devuelve lista de tokens; si False, string (espacio-separado).
    """

    # Regex para limpieza previa
    re_url = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
    re_email = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
    re_mention = re.compile(r"(?<!\w)@\w+")
    re_hashtag = re.compile(r"(?<!\w)#\w+")
    re_currency= re.compile(r"[€$£¥₿]")
    re_space = re.compile(r"\s+")
    re_nonbasic = re.compile(r"[^0-9A-Za-zÁÉÍÓÚÜÑáéíóúüñ\s]", flags=re.UNICODE)

    def _clean_text_basic(txt: str):

        if not isinstance(txt, str):
            return ""

        t = txt.strip()
        t = re_url.sub(" ", t)
        t = re_email.sub(" ", t)
        t = re_mention.sub(" ", t)
        t = re_hashtag.sub(" ", t)
        t = re_currency.sub(" ", t)
        t = t.lower()
        t = re_nonbasic.sub(" ", t)
        t = re_space.sub(" ", t).strip()
        return t

    # Normalización de acentos
    def _strip_accents(txt: str) -> str:
        return unidecode(txt) if txt else txt

    # spaCy para lematizar y filtrar
    nlp = spacy.load(spacy_model, disable=["ner"])

    if nlp.max_length < 2_000_000:
        nlp.max_length = 2_000_000

    #stopwords
    stopwords_spacy = nlp.Defaults.stop_words if remove_stopwords else set()

    def _token_ok(tok):
        if remove_punct and (tok.is_punct or tok.is_space or tok.is_quote):
            return False
        if remove_numbers and (tok.like_num or tok.shape_.isdigit()):
            return False
        if remove_stopwords and tok.lemma_ in stopwords_spacy:
            return False
        if len(tok.lemma_) < min_token_len:
            return False
        return True

    # Pipeline
    cleaned = df[input_col].fillna("").map(_clean_text_basic)
    if remove_accents:
        cleaned = cleaned.map(_strip_accents)

    results = []
    for doc in nlp.pipe(cleaned.tolist(), batch_size=batch_size, n_process=n_process):
        toks = [tok.lemma_.lower() for tok in doc if _token_ok(tok)]
        if return_tokens:
            results.append(toks)
        else:
            results.append(" ".join(toks))

    df = df.copy()
    df[output_col] = results
    return df


In [53]:
noticias1 = preprocess_text_column(noticias,
    input_col="cuerpo",
    output_col="texto_tfidf",
    min_token_len=2,
    batch_size=256,
    return_tokens=False)

In [42]:
noticias1[['cuerpo' , 'texto_tfidf']].head(10)

,cuerpo,texto_tfidf
0,La Secretaría de Gestión y Control Territorial...,secretaria gestion control territorial medelli...
1,Medellín sigue en su esfuerzo por recuperar su...,medellin seguir esfuerzo recuperar parque publ...
2,En un operativo conjunto entre la Fiscalía Gen...,operativo conjunto fiscalia general nacion pol...
3,"El próximo 6 de septiembre, la Nueva Villa de ...",septiembre villa aburra belen sero escenario f...
4,"Miguel Andrés Quintero Calle ríe a carcajadas,...",miguel andr quintero callar rie carcajada hari...
5,La Fiscalía General de la Nación avanza en una...,fiscalia general nacion avanzar delicado inves...
6,La Secretaría de Medio Ambiente de Medellín in...,secretaria ambiente medellin camara trampa ins...
7,"La Administración Distrital, a través del Fond...",administracion distrital trav fondo valorizaci...
8,La Alcaldía de Medellín anunció en la mañana d...,alcaldia medellin anuncio manana miercol inici...
9,Los gremios del transporte público colectivo e...,gremio transporte publico colectivo medellin e...


In [59]:
noticias.to_csv('Corpus_Raw.csv')

In [56]:
noticias1.to_csv('Corpus_Preprocessing.csv')

## **2. Implementación del Embedding**

---

Implemente la estrategia de embedding a partir del conjunto de datos pre-procesado. Recuerde que:

- `sklearn`: permite implementar bolsas de palabras, TF-IDF y bolsas de N-grams a partir del módulo `sklearn.feature_extraction.text`.
- `gensim`: permite implementar word2vec, fasttext y doc2vec desde `gensim.models`.
- `spacy`: permite representar textos con embeddings pre-entrenados con el atributo `vector`.


In [55]:

import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer


# TF-IDF de PALABRAS (unigramas + bigramas)
tfidf_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),      # uni + bi
    min_df=3,max_df=0.9, sublinear_tf=True, norm="l2")

# TF-IDF de CARACTERES (trigramas a pentagramas)
tfidf_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,max_df=0.9,sublinear_tf=True,norm="l2")

# Funcion para hacer inferencia (no hace fit)
def transform_embeddings(df_test, tfidf_word, tfidf_char, col="texto_tfidf"):
    """Transforma con los vectorizadores ya ajustados y concatena."""
    Xw = tfidf_word.transform(df_test[col])
    Xc = tfidf_char.transform(df_test[col])
    X  = sp.hstack([Xw, Xc], format="csr")
    return X

# Funcion para construir los embeddings (con fit)
def fit_transform_embeddings(df_train, col="texto_tfidf"):
    """Ajusta ambos vectorizadores y devuelve la matriz combinada + los objetos."""
    Xw = tfidf_word.fit_transform(df_train[col])
    Xc = tfidf_char.fit_transform(df_train[col])
    X  = sp.hstack([Xw, Xc], format="csr")
    return X, tfidf_word, tfidf_char

X_nlp, v_word, v_char = fit_transform_embeddings(noticias1, col="texto_tfidf")


En este caso lo que hacemos con `sp.hstack([Xw, Xc], format="csr")` es pegar la matriz TF-IDF **caracteres-pentagramas** con la de **palabras-bigramas** de fora horizontal, por eso el resultado terminan siendo 195748 columnas, pero gracias a esto obtenemos un nivel de granularidad muy grande para entrenar cualquier modelo de clasificacion


In [60]:
X_nlp

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 17280716 stored elements and shape (6511, 195748)>

In [46]:
v_word

TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1, 2), sublinear_tf=True)

In [47]:
v_char

TfidfVectorizer(analyzer='char', max_df=0.9, min_df=3, ngram_range=(3, 5),
                sublinear_tf=True)

## **3. Exploración del Embedding**

---

Puede explorar la representación obtenida por medio de distintas técnicas de visualización o métricas:

- **Análisis de Correlaciones**: si tiene una variable objetivo, puede evaluar correlaciones entre los embeddings y dicha variable.
- **Nubes de palabras**: puede utilizar gráficos de tipo `wordcloud` para visualizar representaciones basadas en conteos
- **Distribuciones**: puede calcular histogramas o gráficos de densidad para mostrar la distribución de embeddings semánticos.


In [ ]:
# ---**INGRESE SU CÓDIGO**---

## **Créditos**

- **Profesor:** [Felipe Restrepo Calle](https://dis.unal.edu.co/~ferestrepoca/)
- **Asistentes docentes:**
  - [Juan Sebastián Lara Ramírez](https://www.linkedin.com/in/juan-sebastian-lara-ramirez-43570a214/).
- **Diseño de imágenes:**
  - [Rosa Alejandra Superlano Esquibel](mailto:rsuperlano@unal.edu.co).
- **Coordinador de virtualización:**
  - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - _Facultad de Ingeniería_
